In [ ]:
!pip -q install flask flask-cors pyngrok pandas scikit-learn joblib

In [ ]:
import os
import io
import json
import pandas as pd
import numpy as np

from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

app = Flask(__name__)
CORS(app)

MESES = {
    1: "Ene", 2: "Feb", 3: "Mar", 4: "Abr", 5: "May", 6: "Jun",
    7: "Jul", 8: "Ago", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dic"
}

def franja_horaria(hora):
    try:
        hora = int(hora)
        if 6 <= hora <= 11:
            return "Mañana"
        elif 12 <= hora <= 17:
            return "Tarde"
        else:
            return "Noche"
    except:
        return "Sin dato"

def nivel_riesgo(prob):
    if prob >= 0.70:
        return "Alto"
    elif prob >= 0.40:
        return "Medio"
    return "Bajo"

def preparar_dataset(df):
    df = df.copy()

    columnas_necesarias = [
        "hora", "mes", "anio", "id_edificio", "id_ambiente",
        "ocupacion", "temperatura", "demanda_pico_kw",
        "factor_potencia", "consumo_kwh"
    ]

    for col in columnas_necesarias:
        if col not in df.columns:
            df[col] = 0

    df["hora"] = pd.to_numeric(df["hora"], errors="coerce").fillna(0).astype(int)
    df["mes"] = pd.to_numeric(df["mes"], errors="coerce").fillna(1).astype(int)
    df["anio"] = pd.to_numeric(df["anio"], errors="coerce").fillna(2026).astype(int)
    df["ocupacion"] = pd.to_numeric(df["ocupacion"], errors="coerce").fillna(0)
    df["temperatura"] = pd.to_numeric(df["temperatura"], errors="coerce").fillna(0)
    df["demanda_pico_kw"] = pd.to_numeric(df["demanda_pico_kw"], errors="coerce").fillna(0)
    df["factor_potencia"] = pd.to_numeric(df["factor_potencia"], errors="coerce").fillna(0)
    df["consumo_kwh"] = pd.to_numeric(df["consumo_kwh"], errors="coerce").fillna(0)

    if "nombre_edificio" not in df.columns:
        df["nombre_edificio"] = "Edificio " + df["id_edificio"].astype(str)

    if "tipo_ambiente" not in df.columns:
        df["tipo_ambiente"] = "Ambiente"

    df["nombre_mes"] = df["mes"].map(MESES).fillna("Sin mes")
    df["periodo"] = df["anio"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2)
    df["franja_horaria"] = df["hora"].apply(franja_horaria)

    return df

def ejecutar_modelo_ml(df):
    df = preparar_dataset(df)

    columnas_modelo = [
        "hora", "mes", "anio", "id_edificio", "id_ambiente",
        "ocupacion", "temperatura", "demanda_pico_kw",
        "factor_potencia"
    ]

    X = df[columnas_modelo].copy()
    y_reg = df["consumo_kwh"]

    # Variable objetivo para clasificación de sobreconsumo
    limite_alto = df["consumo_kwh"].quantile(0.75)
    df["sobreconsumo_real"] = (df["consumo_kwh"] >= limite_alto).astype(int)
    y_clf = df["sobreconsumo_real"]

    # Modelo predictivo de consumo
    modelo_regresion = RandomForestRegressor(
        n_estimators=120,
        random_state=42,
        max_depth=8
    )
    modelo_regresion.fit(X, y_reg)

    pred_consumo = modelo_regresion.predict(X)
    df["pred_consumo_kwh"] = np.round(pred_consumo, 2)

    # Modelo de clasificación de riesgo
    modelo_clasificacion = RandomForestClassifier(
        n_estimators=120,
        random_state=42,
        max_depth=8
    )
    modelo_clasificacion.fit(X, y_clf)

    prob_riesgo = modelo_clasificacion.predict_proba(X)[:, 1]
    df["riesgo_sobreconsumo_prob"] = np.round(prob_riesgo, 3)
    df["riesgo_sobreconsumo_pred"] = df["riesgo_sobreconsumo_prob"].apply(nivel_riesgo)

    # Indicadores semánticos / KPIs
    df["eficiencia_energetica_pct"] = np.round(
        np.where(
            df["demanda_pico_kw"] > 0,
            (df["ocupacion"] / df["demanda_pico_kw"]) * 10,
            0
        ),
        2
    )

    df["riesgo_energetico_indice"] = df["riesgo_sobreconsumo_prob"].apply(nivel_riesgo)

    columnas_salida = [
        "id_tiempo", "hora", "mes", "anio", "id_edificio", "id_ambiente",
        "ocupacion", "temperatura", "demanda_pico_kw", "factor_potencia",
        "consumo_kwh", "nombre_edificio", "tipo_ambiente", "nombre_mes",
        "periodo", "franja_horaria", "eficiencia_energetica_pct",
        "riesgo_energetico_indice", "pred_consumo_kwh", "sobreconsumo_real",
        "riesgo_sobreconsumo_prob", "riesgo_sobreconsumo_pred"
    ]

    for col in columnas_salida:
        if col not in df.columns:
            df[col] = ""

    return df[columnas_salida]

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "ok": True,
        "message": "API Colab SmartCampus activa"
    })

@app.route("/predict", methods=["POST"])
def predict():
    if "csv" not in request.files:
        return jsonify({
            "ok": False,
            "error": "No se recibió archivo CSV"
        }), 400

    archivo = request.files["csv"]

    try:
        df = pd.read_csv(archivo)
        df_procesado = ejecutar_modelo_ml(df)

        csv_buffer = io.StringIO()
        df_procesado.to_csv(csv_buffer, index=False)
        csv_texto = csv_buffer.getvalue()

        resumen = {
            "registros": int(len(df_procesado)),
            "consumo_total_kwh": float(df_procesado["consumo_kwh"].sum()),
            "prediccion_total_kwh": float(df_procesado["pred_consumo_kwh"].sum()),
            "riesgo_promedio": float(df_procesado["riesgo_sobreconsumo_prob"].mean())
        }

        return jsonify({
            "ok": True,
            "message": "Predicción generada correctamente",
            "resumen": resumen,
            "csv": csv_texto
        })

    except Exception as e:
        return jsonify({
            "ok": False,
            "error": str(e)
        }), 500

In [ ]:
from pyngrok import ngrok

# Cerrar túneles anteriores
ngrok.kill()

# Pega aquí tu token de ngrok
ngrok.set_auth_token("PEGA_AQUI_TU_TOKEN")

# Crear túnel público hacia Flask
public_url = ngrok.connect(5000).public_url

print("URL pública de Colab:")
print(public_url)

print("Endpoint de predicción:")
print(public_url + "/predict")

# Ejecutar API Flask
app.run(host="0.0.0.0", port=5000)

URL pública de Colab:
https://cupbearer-backed-surgery.ngrok-free.dev
Endpoint de predicción:
https://cupbearer-backed-surgery.ngrok-free.dev/predict
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 01:42:18] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:00:35] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:17:16] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 03:55:57] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [02/Jun/2026 04:03:00] "POST /predict HTTP/1.1" 200 -
